<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 8 - Model Selection and Cross validation</h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

## Objective
In this TD, you will learn about model selection.

In [ ]:
import pandas as pd
from tqdm import trange, tqdm

from rl_models import RLModel, BiasRLModel, RepeatRLModel
from bandit_task import ReversalBanditTask
from matplotlib import pyplot as plt
import seaborn as sns
import scipy
import numpy as np

In [ ]:
%reload_ext autoreload
%autoreload 2

## The task
We will use behavioral data generated with the following `ReversalBanditTask`. It is displayed here so that you understand what we are working with, but since we work with experimental data, we will not need to perform model simulation in this TD.

In [ ]:
np.random.seed(42)
ReversalBanditTask(0.4, 0.6, 0.25, 50).plot()


## Dataset
### Load Participant Data
We will work with a dataset such as the ones you might work with when you run a study with participants. The data is in `experimental_data.csv`. Each line corresponds to a separate **trial**. The columns are the following:
- `subject_id`: the ID of the subject (starting from 1)
- `block`: the block number
- `trial`: the trial number
- `action`: the action taken by the subject (0 or 1)
- `reward`: the reward received by the subject

📝 Load the data and show the first 5 lines

In [ ]:
data = ...
...

### Explore the data
In this section, we will explore the data to understand its structure and the number of trials, blocks, and subjects. This will involve finer mainpulations with the Pandas library. It is an occasion for you to practice your Pandas skills, but ask around to get some tips and make sure you don't spend more than 20 minutes on this section.

📝 Find how many subjects participated in the experiment

In [ ]:
# Your code here

📝 How many blocks were in the experiment?

In [ ]:
# Your code here

📝 How many trials per block?

In [ ]:
# Your code here

### Data processing
The trial-by-trial tabular format might be useful to save the data, but to feed it to RL models, we will need to transform it into a more compact format.

We will extract the data for each subject and stack the blocks together. The resulting data will be a 2D array of shape (n_blocks, n_trials) for both actions and rewards.

Let's try for one example.

📝Create a `subj_1_actions` array that contains the actions of subject 1 for all blocks. The shape of the array should be (8, 100) (8 blocks, 100 trials per block).

> ⚠️ Beware that in the data, the subjects are numbered starting from 1, while in Python, the indexing starts from 0.

In [ ]:
subj_1_actions = ...

Now let's make a helper function to extract the data for a given subject.

📝 Complete the `extract_subject_data` funciton. It should take the subject ID as an argument and return the actions and rewards as 2D arrays of shape (n_blocks, n_trials).

In [ ]:
def extract_subject_data(id: int, data=data):
    subj_data = ... # Extract the data for the given subject
    actions = ...   # Actions stacked by blocks
    rewards = ...   # Rewards stacked by blocks
    return actions, rewards

Verify that the function works as expected by extracting the data for subject 1 and checking the shape of the arrays. They should both have shape (n_blocks, n_trials)

In [ ]:
actions_subj_1, rewards_subj_1 = extract_subject_data(1)
actions_subj_1.shape, rewards_subj_1.shape

## Define the models
We will compare the three variations of the Reinforcement Learning model with which we worked last time. They are already implemented in the `rl_models.py` file and have been imported in this notebook. They were adapted to be able to process blocked data.

The `candidate_models` list below contains the three model classes that you will use for the model recovery. We will reuse it throughout the notebook.

The `model_names` is a list of the model names in string format. It will be used later to display the results of the model recovery in a more readable way.

In [ ]:
candidate_models = [RLModel, BiasRLModel, RepeatRLModel]

model_names = [model.__name__ for model in candidate_models]
model_names

### Cross Validation
#### Step by step

📝 Extract the actions and rewards from subject 1 (number 1, not index 0)

In [ ]:
actions_s1, rewards_s1 = ..., ...

📝 Extract blocks 0, 1, and 2 separately

In [ ]:
actions_s1_b0 = ...
rewards_s1_b0 = ...

actions_s1_b1 = ...
rewards_s1_b1 = ...

actions_s1_b2 = ...
rewards_s1_b2 = ...

⚠️ Make sure that your actions and rewards remain 2D arrays of shape (n_blocks, n_trials), even if you only have one block. You can use the `reshape` method to do this.

In [ ]:
actions_s1_b1.shape, rewards_s1_b1.shape, actions_s1_b2.shape, rewards_s1_b2.shape

📝 Fit a separate `RLModel` to each block's data

In [ ]:
model_b0 = ...
model_b1 = ...
model_b2 = ...


Validation means computing the log likelihood of the model on a block of data that was not used for fitting. The log likelihood is a measure of how well the model explains the data. The higher the log likelihood, the better the model fits the data. We will compare the likelihoods of a model on its fitting data and on unseen data.

📝 Validate all three models on the data from block 0.

In [ ]:
ll_model_b0_val_b0 = ...
ll_model_b1_val_b0 = ...
ll_model_b2_val_b0 = ...

💭 Which model has the best likelihood on data from block 0? Is it surprising?

#### Fitting on many blocks and validating on 1 block
What we usually do when we have many blocks is that we fit the data to a large portion of the blocks, and keep a small portion for validation. This provides a better estimate of the fit. Here, we will use 7 blocks for fitting and 1 block for validation.

📝 Extract again the data from subject 1

In [ ]:
actions_s1 = ...
rewards_s1 = ...

📝 Extract the fitting and validation data. For the exercise, keep block 4 for validation.
> - You can slice numpy arrays to select rows, and use the np.delete() function to remove a specific row from an array
> - Again, check that your arrays remain 2D, even if there is just one block

In [ ]:
fit_actions = ...
fit_rewards = ...

val_actions = ...
val_rewards = ...

📝 Fit an `RLModel` to the fitting data

In [ ]:
fitted_model = ...

📝 Compute the log likelihood on the validation data

In [ ]:
log_likelihood = ...

#### Full cross validation loop
Now let's put all this code in a loop that will perform 8 validations, using each block for validation.

📝 Fill the `cross_validation` function

In [ ]:
def cross_validation(model_class, actions, rewards):
    n_blocks = 8
    cv_log_likelihood = 0
    
    # Loop through the block indexes that will be used for validation
    for val_block in trange(n_blocks, desc=f'Cross-validation {model_class.__name__}', unit='block'):
        # Extract fitting data: all blocks except val_block
        fit_actions = ...
        fit_rewards = ...

        # Extract validation data: only the val_block (make sure the arrays remain 2D)
        val_actions = ...
        val_rewards = ...

        # Fit the model_class to the fitting data
        fitted_model = ...

        # Compute validation log likelihood on the validation data
        log_likelihood = ...
        cv_log_likelihood += log_likelihood

    # The returned likelihood is the sum of all validation block's likelihoods
    return cv_log_likelihood

In [ ]:
cross_validation(RLModel, *extract_subject_data(1))

# Model Selection

### Selection Criteria
Let's reuse the criteria from last TD to compare with Cross Validation.

#### Log likelihood
📝 Complete the `loglikelihood_evaluation` function. It is slightly different from last week because now it takes a model class, that you have to first fit to the data. As we have seen, log likelihood is not an appropriate criterion, but we will use it for comparison.

> ⚠️ Make sure it returns a float, not an array!

In [ ]:
def loglikelihood_evaluation(model_class, actions, rewards) -> float:
    fitted_model = ...
    ll = ...
    return ll

#### Bayesian Information Criterion (BIC)
$$
\text{BIC} = -2 \cdot \log L + k \cdot \log n
$$
Where:
- $L$ is the likelihood of the behavior given the fitted model
- $k$ is the number of parameters in the model
- $n$ is the number of data points

📝 Complete the `bic_evaluation` function.

In [ ]:
def bic_evaluation(model_class, actions, rewards) -> float:
    fitted_model = ...
    ll = ...

    k = ...
    n = ... # Careful! This one is different than last week.
    bic = ...
    return bic

### Akaike Information Criterion (AIC)
$$
\text{AIC} = -2 \cdot \log L + 2k
$$
Where:
- $L$ is the likelihood of the behavior given the fitted model
- $k$ is the number of parameters in the model


📝 Complete the `aic_evaluation` function.

In [ ]:
def aic_evaluation(model_class, actions, rewards) -> float:
    # You can do it
    ...
    
    aic = ...
    return aic

### Model Selection on Subject 1
Now let's perform model selection to find what model best explains the behavior of Subject 1.

📝 Complete the `evaluate_models` function. 
- It sould take a list of model classes to evaluate
- It should take in the actions and rewards of a specific subject (as they are produced by the `extract_subject_data` function).
- It should evaluate each model on all four criteria:
    - log likelihood
    - BIC
    - AIC
    - Cross-validation
- It should return the results in a DataFrame with rows indicating the model names and columns indicating the criteria.

In [ ]:
criteria = ['LL', 'BIC', 'AIC', 'CV']

def evaluate_models(model_classes, actions, rewards):
    # Initializing the DataFrame with appropriate index and column names
    results = pd.DataFrame(index=model_names, columns=criteria)
    
    for model_name, model_class in zip(model_names, model_classes):
        results.loc[model_name, 'LL'] = ...
        results.loc[model_name, 'BIC'] = ...
        results.loc[model_name, 'AIC'] = ...
        results.loc[model_name, 'CV'] = ...
    return results

Run your `evaluate_models` function with data from subject 1.

In [ ]:
eval_s1 = evaluate_models(candidate_models, *extract_subject_data(1))
eval_s1

### Subject-wise model selection
Now let's find out what model best explain each subject's behavior.

#### Evaluate all models on all subjects

We will start by running the `evaluate_models` function that we just wrote on the data from all subjects.

📝 Fill the code below to store the results from all subjects in a list.

In [ ]:
subj_evals = []

subject_ids = data['subject_id'].unique() # List of all subject ids
for subject_id in subject_ids:
    print(f"Evaluating data from subject {subject_id}") 
    
    actions, rewards = ... # Extract the subjet's data
    evaluation = ... # Evaluate the model
    
    subj_evals.append(evaluation)

subj_evals

Now run this cell to pack all this into a nice-looking DataFrame with two levels of column labels.

In [ ]:
subject_results = pd.concat(subj_evals, keys=subject_ids).unstack()
subject_results

#### Select best model on all subjects
We can make these results even easier to read!

📝 Complete the code below select, for each subject and each criterion, the best-scoring model. The resulting `model_selection` DataFrame should be 2-dimensional, with rows corresponding to subjects, columns to criteria, and cells showing the name of the winning model.

In [ ]:
subject_ids = data['subject_id'].unique()
model_selection = pd.DataFrame(index=subject_ids, columns=criteria)
for subject_id in subject_ids:
    model_selection.loc[subject_id, 'LL'] = ...
    model_selection.loc[subject_id, 'BIC'] = ...
    model_selection.loc[subject_id, 'AIC'] = ...
    model_selection.loc[subject_id, 'CV'] = ...

model_selection